In [43]:
!pip install tiktoken

In [15]:
text=''
with open('/content/GOT.txt') as f:
  text = f.read()

In [16]:
import tiktoken
encoding = tiktoken.encoding_for_model("gpt-4")

In [17]:
enc_text=encoding.encode(text)

In [18]:
enc_sample = enc_text[706:756]
context_size=4

In [19]:
x_variable = enc_sample[:context_size]
y_variable = enc_sample[1:context_size+1]

In [20]:
x_variable

[480, 1636, 14618, 704]

In [21]:
y_variable

[1636, 14618, 704, 13]

In [22]:
for i in range(len(x_variable)):
  context=x_variable[0:i+1]
  desired=y_variable[i]
  print('Context ',context, "Desired ---->",desired)


Context  [480] Desired ----> 1636
Context  [480, 1636] Desired ----> 14618
Context  [480, 1636, 14618] Desired ----> 704
Context  [480, 1636, 14618, 704] Desired ----> 13


In [23]:
for i in range(len(x_variable)):
  context=encoding.decode(x_variable[0:i+1])
  desired=encoding.decode([y_variable[i]])
  print(context, "---->",desired)


 G ----> ared
 Gared ---->  pointed
 Gared pointed ---->  out
 Gared pointed out ----> .


In [32]:
import torch
from torch.utils.data import Dataset, DataLoader
import tiktoken


In [33]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        for i in range(0, len(token_ids) - max_length, stride):
            inp = token_ids[i : i + max_length]
            tgt = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(inp))
            self.target_ids.append(torch.tensor(tgt))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]




In [44]:
tokenizer = tiktoken.encoding_for_model("gpt-4")
def create_dataloader(txt, tokenizer,batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    return DataLoader(dataset,
                      batch_size=batch_size,
                      shuffle=shuffle,
                      drop_last=drop_last,
                      num_workers=num_workers)



In [45]:
with open('/content/GOT.txt') as f:
    text = f.read()



In [69]:
dataloader = create_dataloader(text, tokenizer,batch_size=8, max_length=4, stride=4, shuffle=True)
data_iter = iter(dataloader)



In [70]:
for i in range(4):
  first_batch = next(data_iter)

  print('Context',tokenizer.decode(first_batch[0][0].tolist()),'---->',tokenizer.decode(first_batch[1][0].tolist()))


Context Still, I should ----> , I should like
Context  safely out of sight ---->  out of sight,
Context , but Cately ---->  but Catelyn
Context  he stood there and ---->  stood there and looked
